## tl;dr

The retained v3 result is internally consistent and the two failures are real. Yacht's linear response surface reaches the target on its first additional query in all 200 episodes, so a credible successor can only tie it there. Airfoil leaves room for a model-adaptive policy: the preregistered quadratic response surface reaches the target first in 165/200 episodes, while the union of the current Ingot, linear, and quadratic first choices contains a target-setting in 190/200 episodes. This notebook is diagnostic development evidence only; it must not be used as a new external validation result.

## Context & Methods

This notebook audits `latest-results-v3.json` against the frozen CSV fixtures and protocol. It checks dataset grain and target prevalence, recomputes first-query hit counts from retained run identifiers, and measures agreement and oracle-union headroom among the three model-based policies.

### Key Assumptions

- A setting is a target-setting when its retained outcome is at or below the frozen empirical-quantile threshold.
- The oracle union is a diagnostic upper bound, not an implementable policy. It uses hidden outcomes and therefore cannot support a claim.
- v3 has now been inspected and is development evidence for future algorithm work. A future external claim requires different, untouched data and a newly frozen protocol.

In [1]:
from __future__ import annotations

import csv
import json
from pathlib import Path

import numpy as np

root = Path.cwd()
if not (root / 'tools' / 'public-validation').exists():
    root = next(parent for parent in Path.cwd().parents if (parent / 'tools' / 'public-validation').exists())
validation_root = root / 'tools' / 'public-validation'
protocol = json.loads((validation_root / 'protocol-v3.json').read_text(encoding='utf-8'))
result = json.loads((validation_root / 'latest-results-v3.json').read_text(encoding='utf-8'))
dataset_files = {
    'airfoil': validation_root / 'data' / 'airfoil-self-noise.csv',
    'yacht': validation_root / 'data' / 'yacht-hydrodynamics.csv',
}
primary = 'ingot-with-preregistered-mechanism-features'
linear = 'regularized-linear-response-surface'
quadratic = 'regularized-quadratic-response-surface'

## Data

The intended grain is one unique physical setting and retained outcome per row. The frozen integrity contract requires unique identifiers, unique control vectors, finite numeric values, exact fixture hashes, and approximately 15% target prevalence.

In [2]:
profiles = []
outcomes_by_dataset = {}
for record in result['records']:
    dataset = record['dataset']
    with dataset_files[dataset].open(encoding='utf-8', newline='') as stream:
        rows = list(csv.DictReader(stream))
    id_field = 'setting_id'
    outcome_field = next(name for name in rows[0] if name not in protocol['datasets'][dataset]['controls'] and name != id_field)
    identifiers = [row[id_field] for row in rows]
    controls = [tuple(float(row[name]) for name in protocol['datasets'][dataset]['controls']) for row in rows]
    outcomes = {row[id_field]: float(row[outcome_field]) for row in rows}
    outcomes_by_dataset[dataset] = outcomes
    threshold = float(record['target_threshold'])
    profiles.append({
        'dataset': dataset,
        'rows': len(rows),
        'duplicate_ids': len(identifiers) - len(set(identifiers)),
        'duplicate_controls': len(controls) - len(set(controls)),
        'finite_outcomes': sum(np.isfinite(list(outcomes.values()))),
        'target_settings': sum(value <= threshold for value in outcomes.values()),
        'target_rate': sum(value <= threshold for value in outcomes.values()) / len(outcomes),
    })
profiles

[{'dataset': 'airfoil',
  'rows': 1503,
  'duplicate_ids': 0,
  'duplicate_controls': 0,
  'finite_outcomes': np.int64(1503),
  'target_settings': 226,
  'target_rate': 0.15036593479707253},
 {'dataset': 'yacht',
  'rows': 308,
  'duplicate_ids': 0,
  'duplicate_controls': 0,
  'finite_outcomes': np.int64(308),
  'target_settings': 47,
  'target_rate': 0.1525974025974026}]

## Results

First-query hits isolate the decision that matters most in this benchmark because every method already has a high success rate and the minimum possible additional-trial count is one.

In [3]:
diagnostics = []
for record in result['records']:
    dataset = record['dataset']
    threshold = float(record['target_threshold'])
    outcomes = outcomes_by_dataset[dataset]
    first_hits = {method: 0 for method in (primary, linear, quadratic)}
    agreements = {'primary_linear': 0, 'primary_quadratic': 0, 'linear_quadratic': 0, 'all': 0}
    oracle_union = 0
    for episode in record['episodes']:
        selected = {method: episode['methods'][method]['selected_run_ids'][0] for method in first_hits}
        hits = {method: outcomes[run_id] <= threshold for method, run_id in selected.items()}
        for method, hit in hits.items():
            first_hits[method] += int(hit)
        oracle_union += int(any(hits.values()))
        agreements['primary_linear'] += int(selected[primary] == selected[linear])
        agreements['primary_quadratic'] += int(selected[primary] == selected[quadratic])
        agreements['linear_quadratic'] += int(selected[linear] == selected[quadratic])
        agreements['all'] += int(len(set(selected.values())) == 1)
    diagnostics.append({
        'dataset': dataset,
        'episodes': len(record['episodes']),
        'primary_first_hits': first_hits[primary],
        'linear_first_hits': first_hits[linear],
        'quadratic_first_hits': first_hits[quadratic],
        'oracle_union_first_hits': oracle_union,
        **agreements,
    })
diagnostics

[{'dataset': 'airfoil',
  'episodes': 200,
  'primary_first_hits': 148,
  'linear_first_hits': 155,
  'quadratic_first_hits': 165,
  'oracle_union_first_hits': 190,
  'primary_linear': 13,
  'primary_quadratic': 11,
  'linear_quadratic': 68,
  'all': 3},
 {'dataset': 'yacht',
  'episodes': 200,
  'primary_first_hits': 151,
  'linear_first_hits': 200,
  'quadratic_first_hits': 180,
  'oracle_union_first_hits': 200,
  'primary_linear': 10,
  'primary_quadratic': 3,
  'linear_quadratic': 18,
  'all': 1}]

In [4]:
summary = result['summary']
recomputed_means = {}
for record in result['records']:
    unsuccessful = float(record['unsuccessful_trial_value'])
    recomputed_means[record['dataset']] = {}
    for method in (primary, linear, quadratic):
        values = [
            episode['methods'][method]['additional_trials']
            if episode['methods'][method]['additional_trials'] is not None
            else unsuccessful
            for episode in record['episodes']
        ]
        recomputed_means[record['dataset']][method] = float(np.mean(values))
assert recomputed_means == {
    dataset: {method: summary['dataset_method_summaries'][dataset][method]['mean_capped_additional_trials'] for method in (primary, linear, quadratic)}
    for dataset in recomputed_means
}
recomputed_means

{'airfoil': {'ingot-with-preregistered-mechanism-features': 1.525,
  'regularized-linear-response-surface': 1.555,
  'regularized-quadratic-response-surface': 1.36},
 'yacht': {'ingot-with-preregistered-mechanism-features': 1.38,
  'regularized-linear-response-surface': 1.0,
  'regularized-quadratic-response-surface': 1.405}}

### Outcome-blind adaptive response-surface probe

The probe below chooses among raw linear, raw quadratic, and mechanism-augmented linear response surfaces using leave-one-out error on the initial observations only. Candidate outcomes remain hidden until after the first candidate is selected. This is a diagnostic of the proposed model-admission rule, not a tuned or frozen result.

In [5]:
def raw_and_mechanism_features(dataset, rows):
    controls = list(protocol['datasets'][dataset]['controls'])
    bounds = protocol['datasets'][dataset]['controls']
    engineering = {name: np.asarray([float(row[name]) for row in rows]) for name in controls}
    unit = np.column_stack([(engineering[name] - bounds[name][0]) / (bounds[name][1] - bounds[name][0]) for name in controls])
    columns = dict(engineering)
    derived = []
    for feature in protocol['datasets'][dataset]['mechanism_features']:
        inputs = [columns[name] for name in feature['inputs']]
        if feature['operator'] == 'product':
            raw = np.column_stack(inputs).prod(axis=1)
        elif feature['operator'] == 'ratio':
            denominator = np.where(np.abs(inputs[1]) >= 1e-9, inputs[1], 1e-9)
            raw = inputs[0] / denominator
        else:
            raise ValueError('unsupported diagnostic operator: ' + feature['operator'])
        columns[feature['name']] = raw
        derived.append((raw - feature.get('normalization_offset', 0.0)) / feature.get('normalization_scale', 1.0))
    return unit, np.column_stack([unit, *derived])

def quadratic_basis(points):
    values = np.atleast_2d(points)
    return np.column_stack([values, values ** 2, *[values[:, left] * values[:, right] for left in range(values.shape[1]) for right in range(left + 1, values.shape[1])]])

def ridge_coefficients(features, outcomes, ridge):
    design = np.column_stack([np.ones(len(features)), features])
    penalty = np.eye(design.shape[1]) * ridge
    penalty[0, 0] = 0.0
    return np.linalg.solve(design.T @ design + penalty, design.T @ outcomes)

def ridge_predict(train_features, train_outcomes, candidate_features, ridge):
    coefficients = ridge_coefficients(train_features, train_outcomes, ridge)
    return np.column_stack([np.ones(len(candidate_features)), candidate_features]) @ coefficients

def leave_one_out_rmse(features, outcomes, ridge):
    errors = []
    for held_out in range(len(outcomes)):
        retained = np.arange(len(outcomes)) != held_out
        predicted = ridge_predict(features[retained], outcomes[retained], features[[held_out]], ridge)[0]
        errors.append(predicted - outcomes[held_out])
    scale = max(float(np.ptp(outcomes)), 1e-12)
    return float(np.sqrt(np.mean(np.square(errors))) / scale)

adaptive_diagnostics = []
for record in result['records']:
    dataset = record['dataset']
    with dataset_files[dataset].open(encoding='utf-8', newline='') as stream:
        rows = list(csv.DictReader(stream))
    identifiers = [row['setting_id'] for row in rows]
    outcome_field = next(name for name in rows[0] if name not in protocol['datasets'][dataset]['controls'] and name != 'setting_id')
    outcomes = np.asarray([float(row[outcome_field]) for row in rows])
    unit, mechanism = raw_and_mechanism_features(dataset, rows)
    feature_sets = {
        'raw-linear': (unit, 1e-3),
        'raw-quadratic': (quadratic_basis(unit), 1e-2),
        'mechanism-linear': (mechanism, 1e-3),
    }
    model_choices = {name: 0 for name in feature_sets}
    first_hits = 0
    for episode in record['episodes']:
        initial_ids = set(episode['initial_run_ids'])
        observed = np.asarray([index for index, value in enumerate(identifiers) if value in initial_ids])
        candidates = np.asarray([index for index, value in enumerate(identifiers) if value not in initial_ids])
        errors = {name: leave_one_out_rmse(features[observed], outcomes[observed], ridge) for name, (features, ridge) in feature_sets.items()}
        selected_model = min(errors, key=errors.get)
        model_choices[selected_model] += 1
        features, ridge = feature_sets[selected_model]
        predictions = ridge_predict(features[observed], outcomes[observed], features[candidates], ridge)
        selected = candidates[int(np.argmin(predictions))]
        first_hits += int(outcomes[selected] <= float(record['target_threshold']))
    adaptive_diagnostics.append({'dataset': dataset, 'first_hits': first_hits, 'model_choices': model_choices})
adaptive_diagnostics

[{'dataset': 'airfoil',
  'first_hits': 163,
  'model_choices': {'raw-linear': 69,
   'raw-quadratic': 91,
   'mechanism-linear': 40}},
 {'dataset': 'yacht',
  'first_hits': 40,
  'model_choices': {'raw-linear': 0,
   'raw-quadratic': 23,
   'mechanism-linear': 177}}]

In [6]:
def ordinal_rank(values):
    order = np.argsort(values, kind='stable')
    ranks = np.empty(len(values), dtype=float)
    ranks[order] = np.arange(len(values), dtype=float)
    return ranks / max(len(values) - 1, 1)

conservative_probes = []
for record in result['records']:
    dataset = record['dataset']
    with dataset_files[dataset].open(encoding='utf-8', newline='') as stream:
        rows = list(csv.DictReader(stream))
    identifiers = [row['setting_id'] for row in rows]
    outcome_field = next(name for name in rows[0] if name not in protocol['datasets'][dataset]['controls'] and name != 'setting_id')
    outcomes = np.asarray([float(row[outcome_field]) for row in rows])
    unit, _ = raw_and_mechanism_features(dataset, rows)
    raw_models = {'linear': (unit, 1e-3), 'quadratic': (quadratic_basis(unit), 1e-2)}
    hit_counts = {'cv-choice': 0, 'equal-rank-ensemble': 0, 'cv-weighted-rank-ensemble': 0}
    model_choices = {'linear': 0, 'quadratic': 0}
    for episode in record['episodes']:
        initial_ids = set(episode['initial_run_ids'])
        observed = np.asarray([index for index, value in enumerate(identifiers) if value in initial_ids])
        candidates = np.asarray([index for index, value in enumerate(identifiers) if value not in initial_ids])
        predictions = {}
        errors = {}
        for name, (features, ridge) in raw_models.items():
            predictions[name] = ridge_predict(features[observed], outcomes[observed], features[candidates], ridge)
            errors[name] = leave_one_out_rmse(features[observed], outcomes[observed], ridge)
        cv_model = min(errors, key=errors.get)
        model_choices[cv_model] += 1
        choices = {
            'cv-choice': int(np.argmin(predictions[cv_model])),
            'equal-rank-ensemble': int(np.argmin(ordinal_rank(predictions['linear']) + ordinal_rank(predictions['quadratic']))),
        }
        inverse_errors = {name: 1.0 / max(value, 1e-6) ** 2 for name, value in errors.items()}
        weighted_ranks = sum(inverse_errors[name] * ordinal_rank(predictions[name]) for name in raw_models)
        choices['cv-weighted-rank-ensemble'] = int(np.argmin(weighted_ranks))
        for name, position in choices.items():
            selected = candidates[position]
            hit_counts[name] += int(outcomes[selected] <= float(record['target_threshold']))
    conservative_probes.append({'dataset': dataset, **hit_counts, 'cv_model_choices': model_choices})
conservative_probes

[{'dataset': 'airfoil',
  'cv-choice': 161,
  'equal-rank-ensemble': 167,
  'cv-weighted-rank-ensemble': 163,
  'cv_model_choices': {'linear': 81, 'quadratic': 119}},
 {'dataset': 'yacht',
  'cv-choice': 184,
  'equal-rank-ensemble': 192,
  'cv-weighted-rank-ensemble': 189,
  'cv_model_choices': {'linear': 56, 'quadratic': 144}}]

In [7]:
def ranking_stability(features, outcomes, observed, candidates, ridge):
    full_prediction = ridge_predict(features[observed], outcomes[observed], features[candidates], ridge)
    full_rank = ordinal_rank(full_prediction)
    correlations = []
    for held_out in range(len(observed)):
        retained = np.arange(len(observed)) != held_out
        prediction = ridge_predict(features[observed][retained], outcomes[observed][retained], features[candidates], ridge)
        correlation = float(np.corrcoef(full_rank, ordinal_rank(prediction))[0, 1])
        correlations.append(correlation if np.isfinite(correlation) else -1.0)
    return float(np.mean(correlations)), full_prediction

stability_probes = []
for record in result['records']:
    dataset = record['dataset']
    with dataset_files[dataset].open(encoding='utf-8', newline='') as stream:
        rows = list(csv.DictReader(stream))
    identifiers = [row['setting_id'] for row in rows]
    outcome_field = next(name for name in rows[0] if name not in protocol['datasets'][dataset]['controls'] and name != 'setting_id')
    outcomes = np.asarray([float(row[outcome_field]) for row in rows])
    unit, _ = raw_and_mechanism_features(dataset, rows)
    raw_models = {'linear': (unit, 1e-3), 'quadratic': (quadratic_basis(unit), 1e-2)}
    hits = {'most-stable-model': 0, 'stability-weighted-rank': 0}
    choices = {'linear': 0, 'quadratic': 0}
    stability_totals = {'linear': 0.0, 'quadratic': 0.0}
    for episode in record['episodes']:
        initial_ids = set(episode['initial_run_ids'])
        observed = np.asarray([index for index, value in enumerate(identifiers) if value in initial_ids])
        candidates = np.asarray([index for index, value in enumerate(identifiers) if value not in initial_ids])
        model_results = {name: ranking_stability(features, outcomes, observed, candidates, ridge) for name, (features, ridge) in raw_models.items()}
        for name, (stability, _) in model_results.items():
            stability_totals[name] += stability
        selected_model = max(model_results, key=lambda name: model_results[name][0])
        choices[selected_model] += 1
        selected = candidates[int(np.argmin(model_results[selected_model][1]))]
        hits['most-stable-model'] += int(outcomes[selected] <= float(record['target_threshold']))
        combined = sum(max(stability, 0.0) ** 2 * ordinal_rank(prediction) for stability, prediction in model_results.values())
        selected = candidates[int(np.argmin(combined))]
        hits['stability-weighted-rank'] += int(outcomes[selected] <= float(record['target_threshold']))
    stability_probes.append({'dataset': dataset, **hits, 'model_choices': choices, 'mean_stability': {name: value / len(record['episodes']) for name, value in stability_totals.items()}})
stability_probes

[{'dataset': 'airfoil',
  'most-stable-model': 162,
  'stability-weighted-rank': 167,
  'model_choices': {'linear': 107, 'quadratic': 93},
  'mean_stability': {'linear': 0.8858188697446223,
   'quadratic': 0.889150919565269}},
 {'dataset': 'yacht',
  'most-stable-model': 194,
  'stability-weighted-rank': 191,
  'model_choices': {'linear': 147, 'quadratic': 53},
  'mean_stability': {'linear': 0.9741774812177353,
   'quadratic': 0.9634769605892494}}]

In [8]:
linearity_profiles = []
for record in result['records']:
    dataset = record['dataset']
    with dataset_files[dataset].open(encoding='utf-8', newline='') as stream:
        rows = list(csv.DictReader(stream))
    identifiers = [row['setting_id'] for row in rows]
    outcome_field = next(name for name in rows[0] if name not in protocol['datasets'][dataset]['controls'] and name != 'setting_id')
    outcomes = np.asarray([float(row[outcome_field]) for row in rows])
    unit, _ = raw_and_mechanism_features(dataset, rows)
    maxima = []
    for episode in record['episodes']:
        initial_ids = set(episode['initial_run_ids'])
        observed = np.asarray([index for index, value in enumerate(identifiers) if value in initial_ids])
        correlations = [abs(float(np.corrcoef(unit[observed, column], outcomes[observed])[0, 1])) for column in range(unit.shape[1])]
        maxima.append(max(value if np.isfinite(value) else 0.0 for value in correlations))
    linearity_profiles.append({'dataset': dataset, 'max_abs_univariate_correlation_quantiles': dict(zip(['q10', 'q25', 'q50', 'q75', 'q90'], np.quantile(maxima, [0.1, 0.25, 0.5, 0.75, 0.9]).tolist()))})
linearity_profiles

[{'dataset': 'airfoil',
  'max_abs_univariate_correlation_quantiles': {'q10': 0.36470676143908626,
   'q25': 0.45607571900395016,
   'q50': 0.5518968514486923,
   'q75': 0.666813784351964,
   'q90': 0.7710177360253079}},
 {'dataset': 'yacht',
  'max_abs_univariate_correlation_quantiles': {'q10': 0.8034724145935375,
   'q25': 0.8377896524567379,
   'q50': 0.8662537311672893,
   'q75': 0.8948355572549327,
   'q90': 0.9114978946690934}}]

In [9]:
local_search_probes = []
for record in result['records']:
    dataset = record['dataset']
    with dataset_files[dataset].open(encoding='utf-8', newline='') as stream:
        rows = list(csv.DictReader(stream))
    identifiers = [row['setting_id'] for row in rows]
    outcome_field = next(name for name in rows[0] if name not in protocol['datasets'][dataset]['controls'] and name != 'setting_id')
    outcomes = np.asarray([float(row[outcome_field]) for row in rows])
    unit, mechanism = raw_and_mechanism_features(dataset, rows)
    hits = {'nearest-to-best-raw': 0, 'nearest-to-best-mechanism': 0}
    for episode in record['episodes']:
        initial_ids = set(episode['initial_run_ids'])
        observed = np.asarray([index for index, value in enumerate(identifiers) if value in initial_ids])
        candidates = np.asarray([index for index, value in enumerate(identifiers) if value not in initial_ids])
        best = observed[int(np.argmin(outcomes[observed]))]
        for name, features in {'nearest-to-best-raw': unit, 'nearest-to-best-mechanism': mechanism}.items():
            distance = np.linalg.norm(features[candidates] - features[best], axis=1)
            selected = candidates[int(np.argmin(distance))]
            hits[name] += int(outcomes[selected] <= float(record['target_threshold']))
    local_search_probes.append({'dataset': dataset, **hits})
local_search_probes

[{'dataset': 'airfoil',
  'nearest-to-best-raw': 22,
  'nearest-to-best-mechanism': 25},
 {'dataset': 'yacht',
  'nearest-to-best-raw': 114,
  'nearest-to-best-mechanism': 130}]

In [10]:
dominant_linear_probe = []
for record in result['records']:
    dataset = record['dataset']
    with dataset_files[dataset].open(encoding='utf-8', newline='') as stream:
        rows = list(csv.DictReader(stream))
    identifiers = [row['setting_id'] for row in rows]
    outcome_field = next(name for name in rows[0] if name not in protocol['datasets'][dataset]['controls'] and name != 'setting_id')
    outcomes = np.asarray([float(row[outcome_field]) for row in rows])
    unit, _ = raw_and_mechanism_features(dataset, rows)
    quadratic = quadratic_basis(unit)
    hits = 0
    univariate_hits = 0
    linear_choices = 0
    for episode in record['episodes']:
        initial_ids = set(episode['initial_run_ids'])
        observed = np.asarray([index for index, value in enumerate(identifiers) if value in initial_ids])
        candidates = np.asarray([index for index, value in enumerate(identifiers) if value not in initial_ids])
        correlations = [float(np.corrcoef(unit[observed, column], outcomes[observed])[0, 1]) for column in range(unit.shape[1])]
        dominant_column = int(np.argmax(np.abs(correlations)))
        maximum_correlation = abs(correlations[dominant_column])
        use_linear = maximum_correlation >= 0.8
        linear_choices += int(use_linear)
        features, ridge = (unit, 1e-3) if use_linear else (quadratic, 1e-2)
        prediction = ridge_predict(features[observed], outcomes[observed], features[candidates], ridge)
        selected = candidates[int(np.argmin(prediction))]
        hits += int(outcomes[selected] <= float(record['target_threshold']))
        if use_linear:
            feature = unit[:, [dominant_column]]
            prediction = ridge_predict(feature[observed], outcomes[observed], feature[candidates], 1e-3)
        selected = candidates[int(np.argmin(prediction))]
        univariate_hits += int(outcomes[selected] <= float(record['target_threshold']))
    dominant_linear_probe.append({'dataset': dataset, 'multivariate_first_hits': hits, 'univariate_first_hits': univariate_hits, 'linear_choices': linear_choices})
dominant_linear_probe

[{'dataset': 'airfoil',
  'multivariate_first_hits': 164,
  'univariate_first_hits': 163,
  'linear_choices': 11},
 {'dataset': 'yacht',
  'multivariate_first_hits': 198,
  'univariate_first_hits': 198,
  'linear_choices': 182}]

## Takeaways

1. Data integrity is sufficient for the narrow fixed-pool comparison: both fixtures are complete, finite, unique at the declared setting grain, and match the retained target counts.
2. The aggregate failure is not a denominator or aggregation bug. It is driven by model selection: Yacht is effectively a linear-response-surface case at this budget, while Airfoil favors the quadratic baseline more often than the fixed-weight Ingot ensemble.
3. The next production candidate should use outcome-blind, prequential or cross-validated model weighting across GP, linear, quadratic, and mechanism-augmented response surfaces. It must fall back to a simple response surface when that model is clearly more reliable.
4. No v3-derived tuning choice may be promoted as external evidence. Candidate development can use v2, synthetic functions, and v3 as disclosed regression data; final evidence requires newly sourced data and a frozen v4 protocol.